In [4]:
#showing head of the original csv

import pandas as pd
from pathlib import Path
import h3
import numpy as np


In [2]:
PROJECT    = Path(r"E:/tfl_project")
OUT_DIR    = PROJECT / "outputs"

In [2]:
weekly_csv = pd.read_csv("data\\01aJourneyDataExtract10Jan16-23Jan16.csv")

In [3]:
weekly_csv.head()

,Rental Id,Duration,Bike Id,End Date,EndStation Id,EndStation Name,Start Date,StartStation Id,StartStation Name
0,50754225,240,11834,10/01/2016 00:04,383.0,"Frith Street, Soho",10/01/2016 00:00,18,"Drury Lane, Covent Garden"
1,50754226,300,9648,10/01/2016 00:05,719.0,"Victoria Park Road, Hackney Central",10/01/2016 00:00,479,"Pott Street, Bethnal Green"
2,50754227,1200,10689,10/01/2016 00:20,272.0,"Baylis Road, Waterloo",10/01/2016 00:00,425,"Harrington Square 2, Camden Town"
3,50754228,780,8593,10/01/2016 00:14,471.0,"Hewison Street, Old Ford",10/01/2016 00:01,487,"Canton Street, Poplar"
4,50754229,600,8619,10/01/2016 00:11,399.0,"Brick Lane Market, Shoreditch",10/01/2016 00:01,501,"Cephas Street, Bethnal Green"


In [5]:
# Show just the columns that tell the story
cols_to_show = [
 #   "Rental Id",
    "Start Date",
  #  "StartStation Id", 
    "StartStation Name",
    "End Date",
    "EndStation Name",
    "Duration",
]

sample = weekly_csv[cols_to_show].head(5)

# For a Jupyter notebook — renders as a clean HTML table
sample.style.hide(axis="index")

# For copying into a Medium/TDS article — export as markdown
print(sample.to_markdown(index=False))

# For a screenshot — set display options first
pd.set_option("display.max_colwidth", 30)
pd.set_option("display.width", 120)
print(sample.to_string(index=False))

| Start Date       | StartStation Name                | End Date         | EndStation Name                     |   Duration |
|:-----------------|:---------------------------------|:-----------------|:------------------------------------|-----------:|
| 10/01/2016 00:00 | Drury Lane, Covent Garden        | 10/01/2016 00:04 | Frith Street, Soho                  |        240 |
| 10/01/2016 00:00 | Pott Street, Bethnal Green       | 10/01/2016 00:05 | Victoria Park Road, Hackney Central |        300 |
| 10/01/2016 00:00 | Harrington Square 2, Camden Town | 10/01/2016 00:20 | Baylis Road, Waterloo               |       1200 |
| 10/01/2016 00:01 | Canton Street, Poplar            | 10/01/2016 00:14 | Hewison Street, Old Ford            |        780 |
| 10/01/2016 00:01 | Cephas Street, Bethnal Green     | 10/01/2016 00:11 | Brick Lane Market, Shoreditch       |        600 |
      Start Date                StartStation Name         End Date                     EndStation Name  Duration
10/01

In [ ]:
base = pd.read_parquet(OUT_DIR / "base_combined.parquet")

In [10]:
base_head =base.head()

base_head.style.hide(axis="index")

# For copying into a Medium/TDS article — export as markdown
print(base_head.to_markdown(index=False))

|   station_id | trips_start         |   ts |
|-------------:|:--------------------|-----:|
|            1 | 2016-01-10 09:00:00 |    4 |
|            1 | 2016-01-10 10:00:00 |    1 |
|            1 | 2016-01-10 11:00:00 |    2 |
|            1 | 2016-01-10 12:00:00 |    2 |
|            1 | 2016-01-10 13:00:00 |    2 |


In [3]:
base.shape

(9671772, 3)

naive treatment effect, calculated on original data

In [5]:
bf = pd.read_parquet(OUT_DIR / "bike_hourly_with_covariates.parquet")

# Process in chunks to avoid memory spike
chunk_size = 100_000
h3_cells = []
for i in range(0, len(bf), chunk_size):
    chunk = bf.iloc[i:i+chunk_size]
    h3_cells.extend([h3.latlng_to_cell(lat, lon, 8) for lat, lon in zip(chunk["lat"], chunk["lon"])])
    print(f"  Processed {min(i+chunk_size, len(bf)):,} / {len(bf):,}")

bf["h3_cell"] = h3_cells

# Aggregate to cell-day
bf["day"] = pd.to_datetime(bf["trips_start"]).dt.date

cell_day = (
    bf.groupby(["h3_cell", "day"])
    .agg(
        total_trips           = ("ts", "sum"),
        frac_exposed          = ("strike_exposed", "mean"),
        n_stations            = ("station_id", "nunique"),
        temperature_2m        = ("temperature_2m", "mean"),
        precipitation         = ("precipitation", "mean"),
        cloud_cover           = ("cloud_cover", "mean"),
        wind_speed_10m        = ("wind_speed_10m", "mean"),
        is_weekend            = ("is_weekend", "first"),
        is_am_peak            = ("is_am_peak", "first"),
        is_pm_peak            = ("is_pm_peak", "first"),
        is_bank_holiday       = ("is_bank_holiday", "first"),
        is_school_holiday     = ("is_school_holiday", "first"),
        strike_severity_daily_frac = ("strike_severity_daily_frac", "first"),
        days_to_next_strike   = ("days_to_next_strike", "first"),
        days_since_last_strike= ("days_since_last_strike", "first"),
        month                 = ("month", "first"),
        year                  = ("year", "first"),
        doy                   = ("doy", "first"),
        lat                   = ("lat", "mean"),
        lon                   = ("lon", "mean"),
    )
    .reset_index()
)

cell_day["day"]              = pd.to_datetime(cell_day["day"])
cell_day["y_per_station_log1p"] = np.log1p(cell_day["total_trips"] / cell_day["n_stations"])
cell_day["treated"]          = (cell_day["frac_exposed"] > 0).astype(int)

  Processed 100,000 / 9,206,267
  Processed 200,000 / 9,206,267
  Processed 300,000 / 9,206,267
  Processed 400,000 / 9,206,267
  Processed 500,000 / 9,206,267
  Processed 600,000 / 9,206,267
  Processed 700,000 / 9,206,267
  Processed 800,000 / 9,206,267
  Processed 900,000 / 9,206,267
  Processed 1,000,000 / 9,206,267
  Processed 1,100,000 / 9,206,267
  Processed 1,200,000 / 9,206,267
  Processed 1,300,000 / 9,206,267
  Processed 1,400,000 / 9,206,267
  Processed 1,500,000 / 9,206,267
  Processed 1,600,000 / 9,206,267
  Processed 1,700,000 / 9,206,267
  Processed 1,800,000 / 9,206,267
  Processed 1,900,000 / 9,206,267
  Processed 2,000,000 / 9,206,267
  Processed 2,100,000 / 9,206,267
  Processed 2,200,000 / 9,206,267
  Processed 2,300,000 / 9,206,267
  Processed 2,400,000 / 9,206,267
  Processed 2,500,000 / 9,206,267
  Processed 2,600,000 / 9,206,267
  Processed 2,700,000 / 9,206,267
  Processed 2,800,000 / 9,206,267
  Processed 2,900,000 / 9,206,267
  Processed 3,000,000 / 9,206,26

In [6]:
print(f"Naive diff   : {np.expm1(cell_day.loc[cell_day['treated']==1,'y_per_station_log1p'].mean() - cell_day.loc[cell_day['treated']==0,'y_per_station_log1p'].mean())*100:+.1f}%")

Naive diff   : +5.5%
